## NBA Player Position Predictor
### A data science/machine learning project to predict player positions from rate stats

In [ ]:
import nba_api.stats.endpoints as endpoints
print(dir(endpoints))

['AllTimeLeadersGrids', 'AssistLeaders', 'AssistTracker', 'BoxScoreAdvancedV2', 'BoxScoreAdvancedV3', 'BoxScoreDefensiveV2', 'BoxScoreFourFactorsV2', 'BoxScoreFourFactorsV3', 'BoxScoreHustleV2', 'BoxScoreMatchupsV3', 'BoxScoreMiscV2', 'BoxScoreMiscV3', 'BoxScorePlayerTrackV3', 'BoxScoreScoringV2', 'BoxScoreScoringV3', 'BoxScoreSummaryV2', 'BoxScoreSummaryV3', 'BoxScoreTraditionalV2', 'BoxScoreTraditionalV3', 'BoxScoreUsageV2', 'BoxScoreUsageV3', 'CommonAllPlayers', 'CommonPlayerInfo', 'CommonPlayoffSeries', 'CommonTeamRoster', 'CommonTeamYears', 'CumeStatsPlayer', 'CumeStatsPlayerGames', 'CumeStatsTeam', 'CumeStatsTeamGames', 'DefenseHub', 'DraftBoard', 'DraftCombineDrillResults', 'DraftCombineNonStationaryShooting', 'DraftCombinePlayerAnthro', 'DraftCombineSpotShooting', 'DraftCombineStats', 'DraftHistory', 'DunkScoreLeaders', 'FantasyWidget', 'FranchiseHistory', 'FranchiseLeaders', 'FranchisePlayers', 'GLAlumBoxScoreSimilarityScore', 'GameRotation', 'GravityLeaders', 'HomePageLeaders

In [7]:
# now we can see a few options for what to import specifically
from nba_api.stats.endpoints import playerindex 

player_index = playerindex.PlayerIndex(season = '2024-25')
player_index_df = player_index.get_data_frames()[0]

print(player_index_df.columns.tolist())

['PERSON_ID', 'PLAYER_LAST_NAME', 'PLAYER_FIRST_NAME', 'PLAYER_SLUG', 'TEAM_ID', 'TEAM_SLUG', 'IS_DEFUNCT', 'TEAM_CITY', 'TEAM_NAME', 'TEAM_ABBREVIATION', 'JERSEY_NUMBER', 'POSITION', 'HEIGHT', 'WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'ROSTER_STATUS', 'FROM_YEAR', 'TO_YEAR', 'PTS', 'REB', 'AST', 'STATS_TIMEFRAME']


In [ ]:
print(player_index_df['POSITION'].value_counts())
# it seems like not all of the players are present, let's confirm

POSITION
G      65
F      45
C       8
G-F     7
C-F     5
F-C     4
F-G     2
Name: count, dtype: int64


In [ ]:
print(player_index_df.shape)
print(player_index_df['ROSTER_STATUS'].value_counts())
# yup, we're missing most of the league

(136, 26)
ROSTER_STATUS
1.0    102
Name: count, dtype: int64


In [ ]:
player_index_all = playerindex.PlayerIndex()
player_index_all_df = player_index_all.get_data_frames()[0]

print(player_index_all_df.shape)
print(player_index_all_df['POSITION'].value_counts())
# this is the fix

(582, 26)
POSITION
G      243
F      177
C       65
G-F     39
F-C     30
C-F     17
F-G     11
Name: count, dtype: int64


In [ ]:
print(player_index_all_df[['PLAYER_FIRST_NAME', 'PLAYER_LAST_NAME', 'POSITION', 'FROM_YEAR', 'TO_YEAR']].head(20))
# now everyone is up to date for the 2024-25 season

   PLAYER_FIRST_NAME  PLAYER_LAST_NAME POSITION FROM_YEAR TO_YEAR
0           Precious           Achiuwa        F      2020    2025
1             Steven             Adams        C      2013    2025
2                Bam           Adebayo      C-F      2017    2025
3              Ochai            Agbaji        G      2022    2025
4              Santi            Aldama      F-C      2021    2025
5               Trey         Alexander        G      2024    2025
6            Nickeil  Alexander-Walker        G      2019    2025
7            Grayson             Allen        G      2018    2025
8            Jarrett             Allen        C      2017    2025
9               Jose          Alvarado        G      2021    2025
10              Kyle          Anderson      F-G      2014    2025
11              Alex     Antetokounmpo        F      2025    2025
12           Giannis     Antetokounmpo        F      2013    2025
13          Thanasis     Antetokounmpo        F      2015    2025
14        

In [13]:
from nba_api.stats.endpoints import leaguedashplayerstats, playerindex
import pandas as pd

# Pull player stats for the 2024-25 season
stats = leaguedashplayerstats.LeagueDashPlayerStats(season='2024-25')
df = stats.get_data_frames()[0]

print(df.shape)

(569, 67)


In [14]:
# Grab just the columns we need from the position table
positions = player_index_all_df[['PERSON_ID', 'POSITION']]

# Merge with our stats table
df_merged = df.merge(positions, left_on='PLAYER_ID', right_on='PERSON_ID', how='left')

# Check the result
print(df_merged.shape)
print(df_merged['POSITION'].value_counts())

(569, 69)
POSITION
G      186
F      144
C       48
G-F     33
F-C     29
C-F     17
F-G     11
Name: count, dtype: int64


In [ ]:
# the merge worked, but there are about 101 platers that don't have position data
# let's confirm real quick
print(df_merged['POSITION'].isna().sum())

101


In [16]:
# yup, 101 are missing; we should check who these guys are before we do anything
# let's see how many games + minutes they played
df_merged[df_merged['POSITION'].isna()][['PLAYER_NAME', 'MIN', 'GP']].head(20)

,PLAYER_NAME,MIN,GP
7,Adam Flagler,202.965000,37
8,Adama Sanogo,21.498333,4
12,Alec Burks,863.496667,49
14,Alex Ducas,125.341667,21
15,Alex Len,379.790000,46
16,Alex Reese,216.046667,15
31,Anton Watson,22.166667,9
34,Armel Traore,66.666667,9
41,Ben Simmons,1119.693333,51
49,Bol Bol,448.483333,36


In [ ]:
# there are some pretty big names; let's try out another API endpoint to look for positions
from nba_api.stats.endpoints import commonplayerinfo

test = commonplayerinfo.CommonPlayerInfo(player_id=1627732)
test_df = test.get_data_frames()[0]
print(test_df[['DISPLAY_FIRST_LAST', 'POSITION']].head())
# test with Ben Simmons

  DISPLAY_FIRST_LAST       POSITION
0        Ben Simmons  Guard-Forward


In [18]:
import time

# Get the player IDs that are missing position data
missing_ids = df_merged[df_merged['POSITION'].isna()]['PLAYER_ID'].tolist()

# Loop through each one and fetch their position
position_fixes = []

for player_id in missing_ids:
    try:
        info = commonplayerinfo.CommonPlayerInfo(player_id=player_id)
        info_df = info.get_data_frames()[0]
        position = info_df['POSITION'].values[0]
        position_fixes.append({'PLAYER_ID': player_id, 'POSITION_FIX': position})
        time.sleep(0.5)  # pause to avoid hitting the API too fast
    except:
        position_fixes.append({'PLAYER_ID': player_id, 'POSITION_FIX': None})

fixes_df = pd.DataFrame(position_fixes)
print(fixes_df['POSITION_FIX'].value_counts())

POSITION_FIX
Guard             46
Forward           34
Center             6
Guard-Forward      6
Center-Forward     4
Forward-Center     3
Forward-Guard      2
Name: count, dtype: int64


In [19]:
# Merge the fixes into our main dataframe
df_merged = df_merged.merge(fixes_df, on='PLAYER_ID', how='left')

# Fill in missing positions with the fixes
df_merged['POSITION'] = df_merged['POSITION'].fillna(df_merged['POSITION_FIX'])

# Check how many are still missing
print(df_merged['POSITION'].isna().sum())

0


In [ ]:
# great, now everything is merged! But we do have different names for the positions
# some are abbreviated, while others are spelled out fully, so we need to standardize them
print(df_merged['POSITION'].value_counts())

POSITION
G                 186
F                 144
C                  48
Guard              46
Forward            34
G-F                33
F-C                29
C-F                17
F-G                11
Center              6
Guard-Forward       6
Center-Forward      4
Forward-Center      3
Forward-Guard       2
Name: count, dtype: int64
